In [1]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 3, Finished, Available, Finished)

In [2]:
bronze_delta_path = "Tables/rentcast_bronze_delta"
silver_delta_path = "Tables/rentcast_silver_delta"
quarantine_delta_path = "Tables/rentcast_quarantine"
silver_eto_path = "Tables/rentcast_silver_tracking"

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 4, Finished, Available, Finished)

In [3]:
silver_eto_schema = StructType([
    StructField("processing_id", StringType(), False),
    StructField("processing_timestamp", TimestampType(), False),
    StructField("records_read_from_bronze", LongType(), True),
    StructField("records_after_dedup", LongType(), True),
    StructField("records_loaded_to_silver", LongType(), True),
    StructField("records_quarantined", LongType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("processing_duration_seconds", DoubleType(), True)
])

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 5, Finished, Available, Finished)

In [4]:
def normalize_addresses(df):
    
    
    # Step 1: If addressLine1 contains Apt/Unit/Suite/Floor/Room/#, move that part into addressLine2
    df_clean = df.withColumn(
        "addressLine2_temp",
        F.when(
            (F.col("addressLine2").isNull()) &
            F.col("addressLine1").rlike("(?i)(#|Apt|Unit|Suite|Fl|Floor|Room)"),
            F.regexp_extract("addressLine1", "(?i)(#\\s*\\w+|Apt\\s*\\w*|Unit\\s*\\w*|Suite\\s*\\w*|Fl\\s*\\w*|Floor\\s*\\w*|Room\\s*\\w*)", 0)
        ).otherwise(F.col("addressLine2"))
    )
    
    # Step 2: Remove Apt/Unit/Suite/Floor/Room/# from addressLine1
    df_clean = df_clean.withColumn(
        "addressLine1_clean",
        F.regexp_replace("addressLine1", "(?i)(,?\\s*(#\\s*\\w+|Apt\\s*\\w*|Unit\\s*\\w*|Suite\\s*\\w*|Fl\\s*\\w*|Floor\\s*\\w*|Room\\s*\\w*))", "")
    )
    
    # Step 3: Normalize addressLine2 terms
    df_clean = df_clean.withColumn(
        "addressLine2_normalized",
        F.when(F.col("addressLine2_temp").rlike("(?i)^(fl|Fl)\\s"), F.regexp_replace("addressLine2_temp", "(?i)^(fl|Fl)", "Floor"))
         .when(F.col("addressLine2_temp").rlike("(?i)^UNIT\\s"), F.regexp_replace("addressLine2_temp", "(?i)^UNIT", "Unit"))
         .when(F.col("addressLine2_temp").rlike("(?i)^room\\s"), F.regexp_replace("addressLine2_temp", "(?i)^room", "Room"))
         .when(F.col("addressLine2_temp").rlike("^#"), F.regexp_replace("addressLine2_temp", "^#", "Unit"))
         .otherwise(F.col("addressLine2_temp"))
    )
    
    # Step 4: Trim whitespace
    df_clean = df_clean.withColumn(
        "addressLine1",
        F.trim(F.col("addressLine1_clean"))
    ).withColumn(
        "addressLine2",
        F.trim(F.col("addressLine2_normalized"))
    )
    
    # Drop temporary columns
    df_clean = df_clean.drop("addressLine2_temp", "addressLine1_clean", "addressLine2_normalized")
    
    return df_clean

def format_dates(df): 
    date_columns = ['createdDate', 'lastSeenDate', 'listedDate', 'removedDate']
    
    for col_name in date_columns:
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                F.to_date(F.col(col_name))
            )
    
    return df

def format_zipcode(df): 
    df = df.withColumn(
        "zipCode",
        F.when(
            F.col("zipCode").isNotNull(),
            F.lpad(F.substring(F.regexp_replace(F.col("zipCode"), "[^0-9]", ""), 1, 5), 5, "0")
        ).otherwise(None)
    )
    
    return df

def identify_quarantine_records(df): 
    
    df = df.withColumn(
        "quarantine_reasons",
        F.array_remove(
            F.array(
                F.when(F.col("id").isNull(), F.lit("id_is_null")).otherwise(F.lit(None)),
                F.when(F.col("latitude").isNull(), F.lit("latitude_is_null")).otherwise(F.lit(None)),
                F.when(F.col("longitude").isNull(), F.lit("longitude_is_null")).otherwise(F.lit(None)),
                F.when(F.col("price").isNull(), F.lit("price_is_null")).otherwise(F.lit(None))
            ),
            None
        )
    )
    
    df = df.withColumn(
        "is_quarantined",
        F.size(F.col("quarantine_reasons")) > 0
    )
    
    df = df.withColumn(
        "quarantine_reason",
        F.when(
            F.col("is_quarantined"),
            F.concat_ws(", ", F.col("quarantine_reasons"))
        ).otherwise(F.lit(None))
    )
    
    return df

def deduplicate_records(df): 
    
    # Create window partitioned by id, ordered by lastSeenDate descending
    window_spec = Window.partitionBy("id").orderBy(F.desc("lastSeenDate"))
    
    # Add row number
    df_with_row_num = df.withColumn("row_num", F.row_number().over(window_spec))
    
    # Keep only the first row (most recent) for each id
    df_deduped = df_with_row_num.filter(F.col("row_num") == 1).drop("row_num")
    
    return df_deduped

def log_silver_eto_entry(processing_id, records_read, records_after_dedup, records_to_silver, 
                         records_quarantined, status, error_message=None, duration=None):
     
    
    eto_entry = spark.createDataFrame([{
        "processing_id": processing_id,
        "processing_timestamp": datetime.now(),
        "records_read_from_bronze": records_read,
        "records_after_dedup": records_after_dedup,
        "records_loaded_to_silver": records_to_silver,
        "records_quarantined": records_quarantined,
        "status": status,
        "error_message": error_message,
        "processing_duration_seconds": duration
    }], schema=silver_eto_schema)
    
    if DeltaTable.isDeltaTable(spark, silver_eto_path):
        eto_entry.write.format("delta").mode("append").save(silver_eto_path)
    else:
        eto_entry.write.format("delta").mode("overwrite").save(silver_eto_path)

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 6, Finished, Available, Finished)

In [5]:
def process_bronze_to_silver(): 
    
    processing_id = str(uuid.uuid4())
    start_time = datetime.now()
    
    print(f"Starting Silver Layer Processing")
    print(f"Processing ID: {processing_id}")
    print("="*60)
    
    try:
        # 1. Read from Bronze
        print("Step 1: Reading from Bronze layer...")
        df_bronze = spark.read.format("delta").load(bronze_delta_path)
        records_read = df_bronze.count()
        print(f" Read {records_read} records from bronze")
        
        # 2. Filter out records where id is null (before any processing)
        print("\nStep 2: Filtering records with null IDs...")
        df_with_id = df_bronze.filter(F.col("id").isNotNull())
        null_id_count = records_read - df_with_id.count()
        print(f" Filtered out {null_id_count} records with null ID")
        
        # 3. Deduplicate based on id and lastSeenDate
        print("\nStep 3: Deduplicating records...")
        duplicates_before = df_with_id.count()
        df_deduped = deduplicate_records(df_with_id)
        records_after_dedup = df_deduped.count()
        duplicates_removed = duplicates_before - records_after_dedup
        print(f" Removed {duplicates_removed} duplicate records")
        print(f" Kept most recent record for each ID based on lastSeenDate")
        
        # 4. Normalize addresses
        print("\nStep 4: Normalizing addresses...")
        df_normalized = normalize_addresses(df_deduped)
        print(f" Normalized address formats")
        
        # 5. Format dates
        print("\nStep 5: Formatting dates...")
        df_formatted = format_dates(df_normalized)
        print(f" Formatted date columns")
        
        # 6. Format zipcode
        print("\nStep 6: Formatting zip codes...")
        df_zipcode = format_zipcode(df_formatted)
        print(f" Formatted zip codes to 5 digits")
        
        # 7. Select only required columns
        print("\nStep 7: Selecting required columns...")
        selected_columns = [
            'addressLine1', 'addressLine2', 'bathrooms', 'bedrooms', 'city', 'county', 
            'countyFips', 'createdDate', 'daysOnMarket', 'formattedAddress', 'id', 
            'lastSeenDate', 'latitude', 'longitude', 'listedDate', 'listingType', 
            'lotSize', 'mlsName', 'mlsNumber', 'price', 'propertyType', 'removedDate', 
            'squareFootage', 'state', 'stateFips', 'status', 'yearBuilt', 'zipCode',
            'listingAgent_name', 'listingAgent_phone', 'listingAgent_email', 
            'listingAgent_website', 'listingOffice_name', 'listingOffice_phone', 
            'listingOffice_email', 'listingOffice_website', 'extraction_date', 'offset_used', 'run_id', 'bronze_batch_id'  
        ]
        
        df_clean = df_zipcode.select(*selected_columns)
        print(f" Selected {len(selected_columns)} columns")
        
        # 8. Identify quarantine records
        print("\nStep 8: Identifying records for quarantine...")
        df_with_quarantine = identify_quarantine_records(df_clean)
        
        # Split into silver and quarantine
        df_quarantine = df_with_quarantine.filter(F.col("is_quarantined") == True)
        df_silver = df_with_quarantine.filter(F.col("is_quarantined") == False)
        
        records_quarantined = df_quarantine.count()
        records_to_silver = df_silver.count()
        
        print(f" {records_to_silver} records passed quality checks")
        print(f" {records_quarantined} records failed quality checks (quarantined)")
        
        # 9. Show quarantine breakdown
        if records_quarantined > 0:
            print("\n  Quarantine Reasons Breakdown:")
            quarantine_summary = df_quarantine.groupBy("quarantine_reason").count().orderBy(F.desc("count"))
            quarantine_summary.show(truncate=False)
        
        # 10. Write to Silver (drop quarantine columns)
        print("\nStep 9: Writing to Silver layer...")
        df_silver_final = df_silver.drop("quarantine_reasons", "is_quarantined", "quarantine_reason")
        
        if DeltaTable.isDeltaTable(spark, silver_delta_path):
            # Use merge to handle updates based on id
            silver_table = DeltaTable.forPath(spark, silver_delta_path)
            
            silver_table.alias("target").merge(
                df_silver_final.alias("source"),
                "target.id = source.id"
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            
            print(f"Merged {records_to_silver} records into Silver table")
        else:
            df_silver_final.write.format("delta").mode("overwrite").save(silver_delta_path)
            print(f"Created Silver table with {records_to_silver} records")
        
        # 11. Write to Quarantine
        if records_quarantined > 0:
            print("\nStep 10: Writing to Quarantine table...")
            df_quarantine_final = df_quarantine.withColumn("quarantine_timestamp", F.current_timestamp())
            df_quarantine_final = df_quarantine_final.withColumn("processing_id", F.lit(processing_id))
            
            if DeltaTable.isDeltaTable(spark, quarantine_delta_path):
                df_quarantine_final.write.format("delta").mode("append").save(quarantine_delta_path)
            else:
                df_quarantine_final.write.format("delta").mode("overwrite").save(quarantine_delta_path)
            
            print(f"Wrote {records_quarantined} records to Quarantine table")
        
        # 12. Log to ETO
        duration = (datetime.now() - start_time).total_seconds()
        log_silver_eto_entry(
            processing_id, 
            records_read, 
            records_after_dedup,
            records_to_silver, 
            records_quarantined, 
            "SUCCESS", 
            None, 
            duration
        )
        

        return {
            "processing_id": processing_id,
            "records_read": records_read,
            "records_to_silver": records_to_silver,
            "records_quarantined": records_quarantined,
            "success": True
        }
        
    except Exception as e:
        duration = (datetime.now() - start_time).total_seconds()
        error_msg = str(e)
        log_silver_eto_entry( processing_id, 0, 0, 0, 0, "FAILED", error_msg, duration)
        
        print(f"\nSilver processing failed: {error_msg}")
        raise

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 7, Finished, Available, Finished)

In [6]:
if __name__ == "__main__":
    result = process_bronze_to_silver()

StatementMeta(, b69830d6-f590-4383-b097-1f7592259310, 8, Finished, Available, Finished)

Starting Silver Layer Processing
Processing ID: 7517593f-087f-460c-9988-d0573f6bd32c
Step 1: Reading from Bronze layer...
 Read 500 records from bronze

Step 2: Filtering records with null IDs...
 Filtered out 0 records with null ID

Step 3: Deduplicating records...
 Removed 0 duplicate records
 Kept most recent record for each ID based on lastSeenDate

Step 4: Normalizing addresses...
 Normalized address formats

Step 5: Formatting dates...
 Formatted date columns

Step 6: Formatting zip codes...
 Formatted zip codes to 5 digits

Step 7: Selecting required columns...
 Selected 40 columns

Step 8: Identifying records for quarantine...
 500 records passed quality checks
 0 records failed quality checks (quarantined)

Step 9: Writing to Silver layer...
Created Silver table with 500 records
